# ASR production — faster-whisper large-v3
Attach the shared private dataset, select a T4 GPU, enable Internet, and set the Kaggle secret `HF_TOKEN`. Change only `WORKER_SLOT` per account.


In [ ]:
import os
import subprocess
import sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
    "faster-whisper==1.1.1", "ctranslate2==4.5.0", "huggingface-hub==0.27.1",
])
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
# Change only WORKER_SLOT on each Kaggle account.
WORKER_SLOT = 1
WORKER_BATCHES = {
    1: ("batch-01", "batch-09"),
    2: ("batch-02", "batch-03", "batch-04"),
    3: ("batch-05", "batch-08"),
    4: ("batch-06", "batch-07"),
}
WORKER_COUNT = 4
AUDIO_ROOT = "/kaggle/input/Vu165/lastdance-asr/asr/audio"
CATALOG_PATH = "/kaggle/input/Vu165/lastdance-asr/asr/audio"
OUTPUT_ROOT = "/kaggle/working/asr/archives"
HF_REPO_ID = "Vu165/lastdance-asr"


In [ ]:
"""Self-contained Kaggle ASR runtime (pasteable; intentionally no repo imports)."""

from __future__ import annotations

import hashlib
import json
import os
import csv
from pathlib import Path


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _load_state(path: Path) -> dict:
    if not path.exists():
        return {"completed": []}
    return json.loads(path.read_text(encoding="utf-8"))


def _save_state(path: Path, state: dict) -> None:
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(state, indent=2), encoding="utf-8")
    os.replace(temporary, path)


def _video_set_sha256(video_ids: list[str]) -> str:
    digest = hashlib.sha256()
    for video_id in sorted(set(video_ids)):
        digest.update(video_id.encode("utf-8") + b"\n")
    return digest.hexdigest()


def _load_frames(path: str | Path | None) -> dict[str, list[tuple[float, int]]]:
    if path is None:
        return {}
    result: dict[str, list[tuple[float, int]]] = {}
    with Path(path).open(encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            result.setdefault(row["video_id"], []).append(
                (float(row["pts_time"]), int(row["keyframe_uid"]))
            )
    return result


def _nearest(video_id: str, start: float, end: float, frames: dict[str, list[tuple[float, int]]]) -> int:
    values = frames.get(video_id, [])
    if not values:
        raise RuntimeError(f"catalog has no keyframes for {video_id}")
    inside = [row for row in values if start <= row[0] <= end]
    target = (start + end) / 2 if inside else start
    return min(inside or values, key=lambda row: (abs(row[0] - target), row[0]))[1]


def run_production(
    audio_dir: str | Path,
    output_jsonl: str | Path,
    *,
    batch_id: str,
    worker_id: str = "kaggle-01",
    state_path: str | Path | None = None,
    model_name: str = "large-v3",
    catalog_path: str | Path | None = None,
    hf_repo_id: str | None = None,
    hf_token: str | None = None,
    manifest_path: str | Path | None = None,
) -> Path:
    """Transcribe each FLAC once and write one terminal envelope per video."""

    from faster_whisper import WhisperModel

    audio_root, destination = Path(audio_dir), Path(output_jsonl)
    destination.parent.mkdir(parents=True, exist_ok=True)
    state_file = Path(state_path or destination.with_suffix(".state.json"))
    state = _load_state(state_file)
    completed = set(state.get("completed", []))
    model = WhisperModel(model_name, device="cuda", compute_type="float16")
    frames = _load_frames(catalog_path or os.environ.get("ASR_CATALOG_PATH"))
    existing = destination.read_text(encoding="utf-8").splitlines() if destination.exists() else []
    existing_rows = [json.loads(line) for line in existing if line.strip()]
    if any(not isinstance(row, dict) or not row.get("video_id") for row in existing_rows):
        raise RuntimeError("ASR output JSONL contains an invalid existing row")
    seen = {str(row["video_id"]) for row in existing_rows}
    all_rows = list(existing_rows)
    with destination.open("a", encoding="utf-8") as output:
        for audio in sorted(audio_root.glob("*.flac")):
            video_id = audio.stem
            if video_id in seen:
                continue
            try:
                segments, info = model.transcribe(
                    str(audio),
                    language=None,
                    vad_filter=True,
                    condition_on_previous_text=False,
                )
                rows = []
                for segment in segments:
                    text = str(segment.text).strip()
                    if not text:
                        continue
                    language = "vi" if str(info.language).lower().startswith("vi") else "en"
                    rows.append({
                        "video_id": video_id, "segment_id": f"s{len(rows):06d}",
                        "start_time": max(0.0, float(segment.start)),
                        "end_time": max(float(segment.start), float(segment.end)),
                        "transcribed_text": text, "language": language,
                        "keyframe_uid_nearest": _nearest(video_id, float(segment.start), float(segment.end), frames)
                    })
                status = "success" if rows else "silent"
                envelope = {
                    "schema_version": 1, "batch_id": batch_id, "video_id": video_id,
                    "status": status, "engine": "whisper_large_v3",
                    "audio_path": f"asr/audio/{batch_id}/{audio.name}",
                    "audio_sha256": _sha256(audio),
                    "audio_duration_seconds": float(getattr(info, "duration", 0.0) or 0.0),
                    "segments": rows
                }
            except (OSError, RuntimeError, ValueError) as exc:
                envelope = {
                    "schema_version": 1, "batch_id": batch_id, "video_id": video_id,
                    "status": "error", "engine": "whisper_large_v3",
                    "audio_path": f"asr/audio/{batch_id}/{audio.name}",
                    "audio_sha256": _sha256(audio), "audio_duration_seconds": 0.0,
                    "segments": [], "error_code": type(exc).__name__,
                    "error_message": str(exc)[:500]
                }
            output.write(json.dumps(envelope, ensure_ascii=False) + "\n")
            output.flush()
            all_rows.append(envelope)
            completed.add(video_id)
            seen.add(video_id)
            state["completed"] = sorted(completed)
            state["batch_id"], state["worker_id"] = batch_id, worker_id
            _save_state(state_file, state)
    audio_files = list(audio_root.glob("*.flac"))
    expected_ids = {audio.stem for audio in audio_files}
    row_ids = [str(row.get("video_id", "")) for row in all_rows]
    duplicate_videos = len(row_ids) - len(set(row_ids))
    foreign_videos = len(set(row_ids) - expected_ids)
    if duplicate_videos or foreign_videos:
        raise RuntimeError("ASR output JSONL contains duplicate or foreign video IDs")
    output_sha256 = _sha256(destination)
    assigned_video_sha256 = _video_set_sha256([audio.stem for audio in audio_files])
    manifest = {
        "schema_version": 1, "batch_id": batch_id, "worker_id": worker_id,
        "catalog_sha256": _sha256(Path(catalog_path)) if catalog_path else "0" * 64,
        "config_sha256": hashlib.sha256(
            f"{model_name}:float16:cuda:vad_filter:condition_on_previous_text=False".encode()
        ).hexdigest(),
        "assigned_video_sha256": assigned_video_sha256,
        "audio_root": f"asr/audio/{batch_id}",
        "engine": "whisper_large_v3",
        "output_jsonl_path": f"asr/archives/{batch_id}/{destination.name}",
        "output_jsonl_sha256": output_sha256,
        "shard_path": f"asr/archives/{batch_id}/{destination.name}",
        "shard_sha256": output_sha256,
        "expected_video_sha256": assigned_video_sha256,
        "record_count": len(all_rows),
        "success_records": sum(row["status"] == "success" for row in all_rows),
        "silent_records": sum(row["status"] == "silent" for row in all_rows),
        "error_records": sum(row["status"] == "error" for row in all_rows),
        "expected_videos": len(audio_files),
        "processed_videos": len(all_rows),
        "success_videos": sum(row["status"] == "success" for row in all_rows),
        "silent_videos": sum(row["status"] == "silent" for row in all_rows),
        "error_videos": sum(row["status"] == "error" for row in all_rows),
        "duplicate_videos": duplicate_videos,
        "missing_videos": len(expected_ids - set(row_ids)),
        "foreign_videos": foreign_videos,
        "completion_gate_passed": len(all_rows) == len(audio_files)
        and not any(row["status"] == "error" for row in all_rows),
    }
    manifest_file = Path(manifest_path or destination.with_suffix(".manifest.json"))
    manifest_file.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    if hf_repo_id and hf_token:
        from huggingface_hub import CommitOperationAdd, HfApi
        api = HfApi(token=hf_token)
        commit_message = f"ASR archive {batch_id}"
        remote_paths = {
            f"asr/archives/{batch_id}/{destination.name}",
            f"asr/archives/{batch_id}/{manifest_file.name}",
        }
        existing_paths = set(
            api.list_repo_files(repo_id=hf_repo_id, repo_type="dataset")
        )
        present = remote_paths & existing_paths
        if present and present != remote_paths:
            raise RuntimeError("remote ASR archive is partial; refusing overwrite")
        if not present:
            api.create_commit(
                repo_id=hf_repo_id,
                repo_type="dataset",
                operations=[
                    CommitOperationAdd(
                        path_in_repo=f"asr/archives/{batch_id}/{manifest_file.name}",
                        path_or_fileobj=manifest_file,
                    ),
                    CommitOperationAdd(
                        path_in_repo=f"asr/archives/{batch_id}/{destination.name}",
                        path_or_fileobj=destination,
                    ),
                ],
                commit_message=commit_message,
            )
    return destination


if __name__ == "__main__":
    run_production(
        os.environ["ASR_AUDIO_DIR"], os.environ["ASR_OUTPUT_JSONL"],
        batch_id=os.environ.get("ASR_BATCH_ID", "batch-01")
    )


In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

for batch_id in WORKER_BATCHES[WORKER_SLOT]:
    audio_dir = Path(AUDIO_ROOT) / batch_id
    output_dir = Path(OUTPUT_ROOT) / batch_id
    output_dir.mkdir(parents=True, exist_ok=True)
    run_production(
        audio_dir,
        output_dir / "asr-envelope.jsonl",
        batch_id=batch_id,
        worker_id=f"kaggle-{WORKER_SLOT:02d}",
        state_path=output_dir / "batch-checkpoint.json",
        catalog_path=CATALOG_PATH,
        hf_repo_id=HF_REPO_ID,
        hf_token=HF_TOKEN or None,
        manifest_path=output_dir / "manifest.json",
    )
